# ISI=1 Position Bias in Sequence Generation

**Goal**: Understand why ISI=1 events cluster at the beginning of sequences,
especially in short (L=15) and mixed-ISI conditions.

| Section | Question |
|---|---|
| 1. Algorithm Walkthrough | Trace the `generate_one()` algorithm step by step |
| 2. Normalized Position Distributions | Compare position distributions normalized by sequence length |
| 3. Position x ISI Correlation | Quantify the bias: is trial position confounded with ISI? |
| 4. Root Cause | Identify the exact code path that causes the front-loading |
| 5. Candidate Fixes | Test alternative placement strategies |

In [ ]:
import sys, os, random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict

sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')

from utils.sequence_utils import ISISequence, StimulusManager
from utls.toy_experiments import (
    make_compact_multi_isi_sequences,
    infer_trial_isis,
)

In [ ]:
# ---- Shared helpers ----

def classify_positions(seq):
    """For each trial, return (role, isi). role: 'foil'|'first_pres'|'repeat'."""
    first_seen = {}
    repeat_of = {}  # repeat_pos -> first_pos
    for i, stim in enumerate(seq):
        if stim in first_seen:
            repeat_of[i] = first_seen[stim]
        else:
            first_seen[stim] = i

    first_with_repeat = set(repeat_of.values())
    fp_to_rep = {fp: rp for rp, fp in repeat_of.items()}

    roles = []
    for i, stim in enumerate(seq):
        if i in repeat_of:
            isi = i - repeat_of[i] - 1
            roles.append(('repeat', isi))
        elif i in first_with_repeat:
            isi = fp_to_rep[i] - i - 1
            roles.append(('first_pres', isi))
        else:
            roles.append(('foil', -1))
    return roles


def normalized_position_histograms(seq_configs, target_isi=1, n_bins=20):
    """Plot normalized (0-1) position distributions for repeat, first-pres, foil."""
    configs_with_target = {k: v for k, v in seq_configs.items()
                           if target_isi in v['isi_values']}
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    titles = [f'ISI={target_isi} repeat positions',
              f'ISI={target_isi} first-pres positions',
              'Foil positions']
    bins = np.linspace(0, 1, n_bins + 1)

    for name, cfg in configs_with_target.items():
        rep_pos, fp_pos, foil_pos = [], [], []
        for seq in cfg['exps']:
            roles = classify_positions(seq)
            max_len = len(seq)
            for i, (role, isi) in enumerate(roles):
                norm = i / (max_len - 1) if max_len > 1 else 0
                if role == 'repeat' and isi == target_isi:
                    rep_pos.append(norm)
                elif role == 'first_pres' and isi == target_isi:
                    fp_pos.append(norm)
                elif role == 'foil':
                    foil_pos.append(norm)
        axes[0].hist(rep_pos, bins=bins, alpha=0.5, density=True, label=name)
        axes[1].hist(fp_pos, bins=bins, alpha=0.5, density=True, label=name)
        axes[2].hist(foil_pos, bins=bins, alpha=0.5, density=True, label=name)

    for ax, title in zip(axes, titles):
        ax.set_xlabel('Normalized position (0=start, 1=end)')
        ax.set_ylabel('Density')
        ax.set_title(title)
        ax.legend(fontsize=7, frameon=False)
        ax.grid(alpha=0.25)
    fig.suptitle(f'Normalized position distributions (ISI={target_isi})',
                 y=1.03, fontsize=13)
    fig.tight_layout()
    return fig


# Dummy stimulus pool (we only need identifiers, not real audio)
stimulus_pool = [f'stim_{i:04d}' for i in range(500)]

---
## 1. Algorithm Walkthrough

The core algorithm in `ISISequence.generate_one()` works in 4 steps:

1. **Step 1 (greedy forward scan)**: Walk `i = 0 .. length-1`. For each position,
   pick the next ISI from a round-robin over the shuffled ISI list. If both
   `isi_key[i]` and `isi_key[i + isi + 1]` are unassigned, place the pair.

2. **Step 2**: Fill remaining unassigned slots as non-repeats (`-1`).

3. **Step 3 (adjust rep rate)**: If rep rate < 0.5, find upgradeable non-repeat
   pairs and convert them. If > 0.5, remove overrepresented pairs.

4. **Step 4**: Validate that each ISI has `>= min_pairs_per_isi` pairs.

**The bias comes from Step 1**: the greedy forward scan always starts at position 0
and tries to fill from left to right. Small ISIs (like ISI=1) need a gap of only 2,
so they succeed at almost every attempt. By the time the scanner reaches the end,
many early positions are already committed to ISI=1.

Let's trace this directly.

In [ ]:
# Reproduce the Step 1 greedy scan for a single ISI=[1] sequence, L=15

def trace_step1(length, isi_values, seed=42):
    """Reproduce Step 1 of generate_one() with full tracing."""
    rng = random.Random(seed)
    isi_key = [-2] * length
    pos_isi_values = [isi for isi in isi_values if isi >= 0]
    rng.shuffle(pos_isi_values)
    print(f"Shuffled ISI order: {pos_isi_values}")

    pairs_by_isi = defaultdict(list)
    isi_index = 0
    i = 0
    log = []

    while i < length:
        isi = pos_isi_values[isi_index % len(pos_isi_values)]
        isi_index += 1
        j = i + isi + 1
        if j < length and isi_key[i] == -2 and isi_key[j] == -2:
            isi_key[i] = isi
            isi_key[j] = isi
            pairs_by_isi[isi].append((i, j))
            log.append(f"  pos {i:2d}: placed ISI={isi} pair at ({i}, {j})")
        else:
            reason = []
            if j >= length: reason.append(f"j={j} out of bounds")
            if i < length and isi_key[i] != -2: reason.append(f"pos {i} already taken")
            if j < length and isi_key[j] != -2: reason.append(f"pos {j} already taken")
            log.append(f"  pos {i:2d}: FAILED ISI={isi} -> ({i}, {j}) [{', '.join(reason)}]")
        i += 1

    # Fill remaining as non-repeats
    for idx in range(length):
        if isi_key[idx] == -2:
            isi_key[idx] = -1

    return isi_key, pairs_by_isi, log


print("=" * 60)
print("ISI=[1] only, L=15")
print("=" * 60)
isi_key, pairs, log = trace_step1(15, [-1, 1], seed=1000)
for line in log:
    print(line)
print(f"\nISI key after Step 1: {isi_key}")
print(f"Pairs placed: {dict(pairs)}")
n_pairs = sum(len(v) for v in pairs.values())
desired = 15 // 3
print(f"Pairs placed: {n_pairs}, desired: {desired}")
print(f"Need to remove {max(0, n_pairs - desired)} pairs in Step 3")

print("\n" + "=" * 60)
print("ISI=[1,2,4] mixed, L=60")
print("=" * 60)
isi_key2, pairs2, log2 = trace_step1(60, [-1, 1, 2, 4], seed=3000)
for line in log2[:30]:  # first 30 steps
    print(line)
print(f"  ... ({len(log2) - 30} more steps)")
print(f"\nPairs by ISI: { {k: len(v) for k, v in pairs2.items()} }")
print(f"Total pairs: {sum(len(v) for v in pairs2.values())}, desired: {60 // 3}")

In [ ]:
# Visualize the Step 1 placement pattern across many seeds

def collect_step1_positions(length, isi_values, n_seeds=200):
    """Run Step 1 across many seeds and collect pair first-pres positions by ISI."""
    positions_by_isi = defaultdict(list)
    for seed in range(n_seeds):
        _, pairs, _ = trace_step1(length, isi_values, seed=seed)
        for isi_val, pair_list in pairs.items():
            for fp, rp in pair_list:
                positions_by_isi[isi_val].append(fp / (length - 1))
    return positions_by_isi


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
bins = np.linspace(0, 1, 21)

# Panel 1: ISI=[1] L=15, Step 1 only (before rate adjustment)
pos1 = collect_step1_positions(15, [-1, 1])
axes[0].hist(pos1[1], bins=bins, alpha=0.7, density=True, color='tab:blue')
axes[0].set_title('Step 1 only: ISI=[1] L=15\nFirst-pres positions')

# Panel 2: ISI=[1] L=60, Step 1 only
pos2 = collect_step1_positions(60, [-1, 1])
axes[1].hist(pos2[1], bins=bins, alpha=0.7, density=True, color='tab:orange')
axes[1].set_title('Step 1 only: ISI=[1] L=60\nFirst-pres positions')

# Panel 3: ISI=[1,2,4] L=60, Step 1 only
pos3 = collect_step1_positions(60, [-1, 1, 2, 4])
for isi_val in [1, 2, 4]:
    if isi_val in pos3:
        axes[2].hist(pos3[isi_val], bins=bins, alpha=0.5, density=True,
                     label=f'ISI={isi_val}')
axes[2].legend(fontsize=9, frameon=False)
axes[2].set_title('Step 1 only: ISI=[1,2,4] L=60\nFirst-pres positions by ISI')

for ax in axes:
    ax.set_xlabel('Normalized position')
    ax.set_ylabel('Density')
    ax.grid(alpha=0.25)
    ax.axhline(1.0, ls=':', color='red', alpha=0.5, label='uniform')

fig.suptitle('Step 1 greedy scan: where do first-presentations land?\n'
             '(before rep-rate adjustment in Step 3)',
             y=1.05, fontsize=13)
fig.tight_layout()
plt.show()

---
## 2. Normalized Position Distributions (full pipeline)

Now run the **full** `make_compact_multi_isi_sequences` pipeline (Step 1-4 +
`StimulusManager`) and look at normalized position distributions.

In [ ]:
# Generate sequences using the full pipeline
seq_configs = {}

# ISI=[1] short
e, k = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool, isi_values=[1],
    n_sequences=8, length=15, min_pairs_per_isi=5, seed=1000,
)
seq_configs['ISI=[1] L=15'] = {'exps': e, 'keys': k, 'isi_values': [1]}

# ISI=[1] long
e, k = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool, isi_values=[1],
    n_sequences=10, length=60, min_pairs_per_isi=5, seed=1001,
)
seq_configs['ISI=[1] L=60'] = {'exps': e, 'keys': k, 'isi_values': [1]}

# ISI=[1,2,4] mixed
e, k = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool, isi_values=[1, 2, 4],
    n_sequences=10, length=60, min_pairs_per_isi=5, seed=3000,
)
seq_configs['ISI=[1,2,4] L=60'] = {'exps': e, 'keys': k, 'isi_values': [1, 2, 4]}

# ISI=[1] intermediate lengths
for L in [30, 45]:
    e, k = make_compact_multi_isi_sequences(
        stimulus_pool=stimulus_pool, isi_values=[1],
        n_sequences=10, length=L, min_pairs_per_isi=5, seed=5000 + L,
    )
    seq_configs[f'ISI=[1] L={L}'] = {'exps': e, 'keys': k, 'isi_values': [1]}

for name, cfg in seq_configs.items():
    print(f"{name}: {len(cfg['exps'])} seqs x {len(cfg['exps'][0])} trials")

In [ ]:
# Normalized position distributions for ISI=1
fig = normalized_position_histograms(seq_configs, target_isi=1)
plt.show()

In [ ]:
# Overlay all ISI=[1]-only configs at different lengths to see the length effect

isi1_only = {k: v for k, v in seq_configs.items()
             if v['isi_values'] == [1]}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bins = np.linspace(0, 1, 21)

for name, cfg in sorted(isi1_only.items(), key=lambda x: len(x[1]['exps'][0])):
    rep_pos, fp_pos = [], []
    for seq in cfg['exps']:
        roles = classify_positions(seq)
        L = len(seq)
        for i, (role, isi) in enumerate(roles):
            norm = i / (L - 1) if L > 1 else 0
            if role == 'repeat' and isi == 1:
                rep_pos.append(norm)
            elif role == 'first_pres' and isi == 1:
                fp_pos.append(norm)
    axes[0].hist(rep_pos, bins=bins, alpha=0.4, density=True, label=name)
    axes[1].hist(fp_pos, bins=bins, alpha=0.4, density=True, label=name)

for ax, title in zip(axes, ['ISI=1 repeat positions', 'ISI=1 first-pres positions']):
    ax.axhline(1.0, ls=':', color='red', alpha=0.5, label='uniform')
    ax.set_xlabel('Normalized position (0=start, 1=end)')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend(fontsize=7, frameon=False)
    ax.grid(alpha=0.25)

fig.suptitle('Does sequence length reduce position bias?', y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

---
## 3. Position x ISI Correlation

Quantify the confound: Spearman correlation between trial position and ISI value.

In [ ]:
from scipy.stats import spearmanr, ks_2samp

print(f"{'Config':<25s} {'rho':>8s} {'p-value':>10s} {'mean_norm_pos':>14s} {'KS vs uniform':>14s}")
print('-' * 75)

for name, cfg in seq_configs.items():
    positions, isis = [], []
    norm_rep_pos = []
    for seq in cfg['exps']:
        roles = classify_positions(seq)
        L = len(seq)
        for i, (role, isi) in enumerate(roles):
            if role == 'repeat':
                positions.append(i)
                isis.append(isi)
            if role == 'repeat' and isi == 1:
                norm_rep_pos.append(i / (L - 1) if L > 1 else 0)

    if len(set(isis)) > 1:
        rho, pval = spearmanr(positions, isis)
    else:
        rho, pval = float('nan'), float('nan')

    # KS test: are ISI=1 repeat positions uniformly distributed?
    if norm_rep_pos:
        uniform_samples = np.linspace(0, 1, len(norm_rep_pos))
        ks_stat, ks_p = ks_2samp(norm_rep_pos, uniform_samples)
        ks_str = f"{ks_stat:.3f} (p={ks_p:.3f})"
        mean_pos = f"{np.mean(norm_rep_pos):.3f}"
    else:
        ks_str = 'N/A'
        mean_pos = 'N/A'

    print(f"{name:<25s} {rho:>8.3f} {pval:>10.4f} {mean_pos:>14s} {ks_str:>14s}")

print("\nIf mean_norm_pos >> 0.5, repeats are late-biased; << 0.5, early-biased.")
print("KS p < 0.05 means positions are significantly non-uniform.")

---
## 4. Root Cause Analysis

The greedy forward scan in Step 1 uses a **linear sweep from position 0**.
For single-ISI conditions, every attempt uses the same ISI, so pairs fill in
a regular pattern starting from position 0. For mixed conditions, the round-robin
cycling means ISI=1 gets tried every N-th position (N = number of ISI values).

The key question: **does Step 3 (rep-rate adjustment) fix or worsen this?**

When Step 1 places too many pairs (common for short sequences with small ISIs),
Step 3 removes excess pairs. But removal uses `shuffle` + iteration, which
doesn't explicitly correct for position bias.

When Step 1 places too few, Step 3 upgrades non-repeat slots. The upgrade
search `find_upgradeable_pairs` scans from position 0, so upgrades also
skew early.

In [ ]:
# Compare Step 1 alone vs full pipeline (Step 1-4)
# to see if Step 3 improves or worsens the bias

def mean_norm_position_step1(length, isi_values, n_seeds=200):
    """Mean normalized first-pres position from Step 1 only."""
    all_norm = []
    for seed in range(n_seeds):
        _, pairs, _ = trace_step1(length, isi_values, seed=seed)
        for isi_val, pair_list in pairs.items():
            if isi_val == 1:  # only ISI=1
                for fp, rp in pair_list:
                    all_norm.append(fp / (length - 1))
    return np.mean(all_norm) if all_norm else float('nan')


def mean_norm_position_full(seq_configs_dict, target_isi=1):
    """Mean normalized repeat position from full pipeline."""
    results = {}
    for name, cfg in seq_configs_dict.items():
        all_norm = []
        for seq in cfg['exps']:
            roles = classify_positions(seq)
            L = len(seq)
            for i, (role, isi) in enumerate(roles):
                if role == 'repeat' and isi == target_isi:
                    all_norm.append(i / (L - 1))
        results[name] = np.mean(all_norm) if all_norm else float('nan')
    return results


print("Mean normalized ISI=1 repeat position (0.5 = perfectly uniform)")
print(f"{'Config':<25s} {'Step 1 only':>12s} {'Full pipeline':>14s} {'Bias direction':>16s}")
print('-' * 70)

full_results = mean_norm_position_full(seq_configs)

step1_configs = [
    ('ISI=[1] L=15', 15, [-1, 1]),
    ('ISI=[1] L=30', 30, [-1, 1]),
    ('ISI=[1] L=45', 45, [-1, 1]),
    ('ISI=[1] L=60', 60, [-1, 1]),
    ('ISI=[1,2,4] L=60', 60, [-1, 1, 2, 4]),
]

for name, length, isi_vals in step1_configs:
    s1_mean = mean_norm_position_step1(length, isi_vals)
    full_mean = full_results.get(name, float('nan'))
    direction = 'EARLY' if full_mean < 0.45 else ('LATE' if full_mean > 0.55 else 'OK')
    print(f"{name:<25s} {s1_mean:>12.3f} {full_mean:>14.3f} {direction:>16s}")

In [ ]:
# Show that find_upgradeable_pairs scans from position 0
# so upgrades tend to land early

def find_upgradeable_pairs(isi_key, isi_value):
    """Exact copy of the inner function from generate_one()."""
    upgrades = []
    for i in range(len(isi_key) - isi_value - 1):
        j = i + isi_value + 1
        if isi_key[i] == -1 and isi_key[j] == -1:
            upgrades.append((i, j))
    return upgrades


# Simulate: start with an isi_key that has some non-repeats, find upgrades
print("Upgrade candidate positions for ISI=1 in a partially-filled L=15 key:")
example_key = [1, -1, 1, -1, 1, -1, 1, -1, -1, -1, 1, -1, 1, -1, -1]
print(f"Key: {example_key}")
candidates = find_upgradeable_pairs(example_key, 1)
print(f"Upgradeable pairs: {candidates}")
print(f"-> All upgrade candidates are at positions {[c[0] for c in candidates]}")
print("\nNote: the code takes candidates[-1] after shuffling,")
print("which is random but still drawn from this position-biased pool.")

---
## 5. Candidate Fixes

Three approaches to eliminate the position bias:

1. **Post-hoc shuffle**: After generating the ISI key, randomly permute
   the positions while preserving ISI distances. (Hard to do correctly.)

2. **Random-start scan**: Instead of always starting at position 0,
   randomize the starting position of the greedy scan.

3. **Two-pass placement**: First randomly choose pair positions (uniformly),
   then assign ISI values to pairs. This decouples position from ISI.

In [ ]:
# Fix 2: Random-start scan

def trace_step1_random_start(length, isi_values, seed=42):
    """Step 1 but starting at a random position and wrapping around."""
    rng = random.Random(seed)
    isi_key = [-2] * length
    pos_isi_values = [isi for isi in isi_values if isi >= 0]
    rng.shuffle(pos_isi_values)

    pairs_by_isi = defaultdict(list)
    isi_index = 0

    # Random starting position
    start = rng.randint(0, length - 1)
    positions = [(start + offset) % length for offset in range(length)]

    for i in positions:
        isi = pos_isi_values[isi_index % len(pos_isi_values)]
        isi_index += 1
        j = i + isi + 1
        if j < length and isi_key[i] == -2 and isi_key[j] == -2:
            isi_key[i] = isi
            isi_key[j] = isi
            pairs_by_isi[isi].append((i, j))
        i += 1

    for idx in range(length):
        if isi_key[idx] == -2:
            isi_key[idx] = -1

    return isi_key, pairs_by_isi


# Fix 3: Slot-first placement

def trace_step1_slot_first(length, isi_values, seed=42):
    """Place pairs by first choosing random positions, then assigning ISIs."""
    rng = random.Random(seed)
    pos_isi_values = [isi for isi in isi_values if isi >= 0]
    desired_pairs = length // 3

    isi_key = [-1] * length  # start all as non-repeat
    pairs_by_isi = defaultdict(list)

    # Shuffle all positions
    all_positions = list(range(length))
    rng.shuffle(all_positions)

    placed = 0
    used = set()

    for fp in all_positions:
        if placed >= desired_pairs:
            break
        if fp in used:
            continue

        # Try each ISI in random order
        rng.shuffle(pos_isi_values)
        for isi in pos_isi_values:
            rp = fp + isi + 1
            if rp < length and rp not in used and fp not in used:
                isi_key[fp] = isi
                isi_key[rp] = isi
                pairs_by_isi[isi].append((fp, rp))
                used.add(fp)
                used.add(rp)
                placed += 1
                break

    return isi_key, pairs_by_isi


# Compare the three approaches
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
bins = np.linspace(0, 1, 21)
n_seeds = 300

configs_to_test = [
    ('Original (forward scan)', trace_step1),
    ('Fix: Random start', trace_step1_random_start),
    ('Fix: Slot-first', trace_step1_slot_first),
]

length = 15
isi_vals = [-1, 1]

for ax_idx, (label, func) in enumerate(configs_to_test):
    all_fp_norm = []
    all_rp_norm = []
    for seed in range(n_seeds):
        _, pairs = func(length, isi_vals, seed=seed)
        for isi_val, pair_list in pairs.items():
            if isi_val >= 0:
                for fp, rp in pair_list:
                    all_fp_norm.append(fp / (length - 1))
                    all_rp_norm.append(rp / (length - 1))

    axes[ax_idx].hist(all_fp_norm, bins=bins, alpha=0.5, density=True,
                      label='First-pres', color='tab:blue')
    axes[ax_idx].hist(all_rp_norm, bins=bins, alpha=0.5, density=True,
                      label='Repeat', color='tab:orange')
    axes[ax_idx].axhline(1.0, ls=':', color='red', alpha=0.5)
    axes[ax_idx].set_title(f'{label}\nISI=[1] L={length}', fontsize=11)
    axes[ax_idx].set_xlabel('Normalized position')
    axes[ax_idx].set_ylabel('Density')
    axes[ax_idx].legend(fontsize=8, frameon=False)
    axes[ax_idx].grid(alpha=0.25)

    mean_fp = np.mean(all_fp_norm)
    mean_rp = np.mean(all_rp_norm)
    axes[ax_idx].text(0.02, 0.95, f'mean FP={mean_fp:.3f}\nmean RP={mean_rp:.3f}',
                      transform=axes[ax_idx].transAxes, fontsize=9, va='top',
                      bbox=dict(boxstyle='round', fc='white', alpha=0.8))

fig.suptitle(f'Candidate fixes for position bias (L={length}, ISI=[1])',
             y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Same comparison but for the mixed ISI=[1,2,4] condition

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
bins = np.linspace(0, 1, 21)

length = 60
isi_vals = [-1, 1, 2, 4]
target_isis = [1, 2, 4]

for col, (label, func) in enumerate(configs_to_test):
    positions_by_isi = defaultdict(list)
    for seed in range(n_seeds):
        _, pairs = func(length, isi_vals, seed=seed)
        for isi_val, pair_list in pairs.items():
            if isi_val >= 0:
                for fp, rp in pair_list:
                    positions_by_isi[isi_val].append(fp / (length - 1))

    for row, target in enumerate(target_isis):
        ax = axes[row, col]
        if target in positions_by_isi:
            data = positions_by_isi[target]
            ax.hist(data, bins=bins, alpha=0.7, density=True,
                    color=f'C{row}')
            mean_pos = np.mean(data)
            ax.text(0.02, 0.95, f'mean={mean_pos:.3f}',
                    transform=ax.transAxes, fontsize=9, va='top',
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))
        ax.axhline(1.0, ls=':', color='red', alpha=0.5)
        ax.set_xlabel('Normalized position')
        ax.set_ylabel('Density')
        ax.grid(alpha=0.25)

        if row == 0:
            ax.set_title(label, fontsize=11)
        if col == 0:
            ax.set_ylabel(f'ISI={target}\nDensity')

fig.suptitle(f'Candidate fixes: ISI=[1,2,4] L={length}\n'
             'First-presentation positions by ISI',
             y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Summary table: bias magnitude for each approach

print("Bias summary: mean normalized first-pres position (0.5 = uniform)")
print(f"{'Approach':<25s} {'ISI=[1] L=15':>14s} {'ISI=[1] L=60':>14s} {'ISI=[1,2,4] L=60':>18s}")
print('-' * 75)

test_cases = [
    (15, [-1, 1], 'ISI=[1] L=15'),
    (60, [-1, 1], 'ISI=[1] L=60'),
    (60, [-1, 1, 2, 4], 'ISI=[1,2,4] L=60'),
]

for label, func in configs_to_test:
    row_vals = []
    for length, isi_vals, case_name in test_cases:
        all_norm = []
        for seed in range(n_seeds):
            _, pairs = func(length, isi_vals, seed=seed)
            for isi_val, pair_list in pairs.items():
                if isi_val == 1:  # ISI=1 only
                    for fp, rp in pair_list:
                        all_norm.append(fp / (length - 1))
        mean_val = np.mean(all_norm) if all_norm else float('nan')
        row_vals.append(f"{mean_val:.3f}")
    print(f"{label:<25s} {row_vals[0]:>14s} {row_vals[1]:>14s} {row_vals[2]:>18s}")

---
## Conclusions

**Fill in after running:**

1. **Root cause**: The greedy forward scan in `ISISequence.generate_one()` Step 1
   starts at position 0 and walks right. Small ISIs succeed at early positions
   before the sequence fills up, creating systematic front-loading.

2. **Step 3 does NOT fix this**: The rep-rate adjustment phase inherits the
   position bias from Step 1, and `find_upgradeable_pairs` also scans from 0.

3. **Short sequences (L=15) are worst**: With few positions, the forward scan
   fills most of the sequence in one pass, leaving little room for redistribution.

4. **Best fix**: [slot-first / random-start / both?] reduces the mean normalized
   position from ___ to ___, achieving near-uniform placement.

**Implication for model fitting**: The position bias means ISI=1 events in short
sequences encounter a small memory bank (early in the trial), which changes the
FA distribution and can produce spurious non-monotonicity in d' vs sigma curves.